# ForgeAI Model Report Lab

This notebook is designed for both Google Colab demonstration and academic reporting. It documents the AI planning model used in ForgeAI, runs the training pipeline, validates the model with tests, evaluates readiness metrics, and shows sample AI coach outputs.

Project topic: AI application for managing and planning home gym workouts.

## 0. Problem Statement

The goal of this model is to generate a personalized home gym plan from onboarding and feedback data.

Inputs:
- Fitness goal: lose fat, build muscle, get stronger, stay active
- Available equipment: dumbbells, bench, yoga mat, bodyweight, etc.
- Body profile: height, weight, age, activity level
- Schedule constraints: training days per week and session duration
- Safety constraints: injuries, soreness areas, allergies, fatigue, missed workouts

Outputs:
- Ranked workout recommendations
- Weekly training and recovery schedule
- Four-week progression plan with deload
- Meal plan and macro targets
- Adaptive coach notes based on user feedback

## 1. Model Summary

This system uses a hybrid AI approach.

Machine learning models:
- Workout ranking: `RandomForestRegressor` from scikit-learn
- Nutrition macro prediction: `MultiOutputRegressor(RandomForestRegressor)`

Rule-based coach layer:
- Equipment filtering
- Injury and soreness filtering
- Allergy and food quality filtering
- Weekly schedule generation
- Four-week progression and deload
- Adaptive intensity adjustment from feedback

Reason for hybrid design: the ML model scores and ranks recommendations, while deterministic rules enforce safety, structure, and product-specific coaching behavior.

## 2. Dataset

The model uses two local datasets.

Workout dataset: `megaGymDataset.csv`
- Exercise title
- Description
- Exercise type
- Body part
- Equipment
- Difficulty level
- Rating

Nutrition dataset: `foods_usda.csv`
- Food keyword and description
- Calories
- Protein, carbs, fat
- Fiber, sugar, sodium
- Food category
- Allergens
- Diet tags
- Serving metadata

In [ ]:
# If needed, change this path to the folder that contains recommender.py.
import os
from pathlib import Path

candidate_paths = [Path('/content/model_ai'), Path.cwd()]
for path in candidate_paths:
    if (path / 'recommender.py').exists():
        os.chdir(path)
        break

print('Working directory:', Path.cwd())

In [ ]:
!pip install -q numpy scikit-learn joblib

In [ ]:
import csv
from collections import Counter
from pathlib import Path

def dataset_summary(path):
    with Path(path).open(encoding='utf-8-sig', newline='') as handle:
        reader = csv.DictReader(handle)
        rows = list(reader)
    return len(rows), reader.fieldnames, rows

workout_count, workout_columns, workout_rows = dataset_summary('megaGymDataset.csv')
food_count, food_columns, food_rows = dataset_summary('foods_usda.csv')

{
    'workout_rows': workout_count,
    'workout_columns': workout_columns,
    'food_rows': food_count,
    'food_columns': food_columns,
}

## 3. Feature Engineering and Training Technique

Training type: supervised regression.

Workout feature vector:
- Rating
- Beginner, intermediate, expert flags
- Strength-training flag
- Bodyweight flag
- Dumbbell flag
- Goal-body-part match
- Equipment match

Workout target generation:
- Base exercise rating
- Bonus for beginner-friendly level
- Bonus when body part matches the user's goal
- Bonus when equipment matches the available equipment pool
- Bonus for strength exercises

Nutrition target generation:
- BMR and activity factor estimate daily calories
- Goal-specific macro ratios estimate protein, carbs, and fat
- Multi-output regression predicts the three macro targets

## 4. Run Tests

The tests validate the ML artifact pipeline, inference contract, safety filters, meal plan, progression plan, adaptive feedback, and evaluation metrics.

In [ ]:
!python -m unittest -v test_recommender.py

## 5. Train Model

This regenerates:
- `recommender_artifacts.json`
- `models/workout_regressor.joblib`
- `models/nutrition_regressor.joblib`

The `.joblib` files store the trained scikit-learn models for real persisted inference.

In [ ]:
!python train_recommender.py

## 6. Evaluate Readiness

The readiness score is an internal MVP regression metric. It is not a clinical benchmark.

Metrics:
- `equipment_match_rate`: recommended workouts match available equipment
- `coachable_description_rate`: workouts have usable descriptions
- `beginner_safe_rate`: workouts are beginner/intermediate safe
- `coach_contract_score`: output includes required AI coach fields
- `adaptation_score`: plan adapts to fatigue, missed workouts, and soreness
- `overall_readiness_score`: aggregate score

In [ ]:
from pathlib import Path
from recommender import RecommenderArtifacts, evaluate_recommender

artifacts = RecommenderArtifacts.load(Path('recommender_artifacts.json'))
evaluate_recommender(artifacts)

## 7. Sample Inference

This section creates a user profile and generates a full AI coach plan.

In [ ]:
from recommender import OnboardingProfile, PlanFeedback, recommend_plan

profile = OnboardingProfile(
    goal='lose fat',
    equipment=['Dumbbells', 'Bench'],
    height_cm=175,
    weight_kg=70,
    age=25,
    activity_level='active',
    training_days_per_week=4,
    session_minutes=45,
    experience_level='beginner',
)

feedback = PlanFeedback(
    missed_workouts=1,
    fatigue_level='normal',
    soreness_areas=[],
)

plan = recommend_plan(profile, artifacts, feedback=feedback)
plan['coach_summary']

In [ ]:
plan['weekly_schedule']

In [ ]:
plan['progression_plan']

In [ ]:
plan['workouts'][:2]

In [ ]:
plan['meal_plan']

## 8. Adaptive Feedback Demo

This simulates a user who missed workouts and feels tired. The model should reduce intensity and prioritize recovery.

In [ ]:
tired_feedback = PlanFeedback(
    missed_workouts=2,
    fatigue_level='high',
    soreness_areas=['shoulder'],
)

adaptive_plan = recommend_plan(profile, artifacts, feedback=tired_feedback)
{
    'readiness_adjustment': adaptive_plan['readiness_adjustment'],
    'coach_notes': adaptive_plan['coach_notes'],
    'first_three_days': adaptive_plan['weekly_schedule'][:3],
}

## 9. Inference Pipeline

The inference pipeline works as follows:

1. Validate user profile and safety constraints.
2. Normalize goal and activity level.
3. Build workout feature vectors from the exercise dataset.
4. Predict workout scores using the persisted Random Forest model.
5. Apply rule-based filters for equipment, injuries, soreness, and beginner safety.
6. Select workouts and attach sets, reps, rest, rationale, and substitutions.
7. Predict macro targets using the nutrition model.
8. Build a three-meal nutrition plan with food quality and allergy filters.
9. Build weekly schedule and four-week progression plan.
10. Adjust intensity and coach notes based on user feedback.

## 10. Strengths and Limitations

Strengths:
- Uses persisted machine learning models for inference
- Produces a complete structured coach plan
- Supports workout, nutrition, progression, recovery, and adaptive feedback
- Includes safety filters for injuries, soreness, allergies, and unsafe profile values
- Has automated tests and evaluation metrics

Limitations:
- It is not a conversational LLM coach
- It does not analyze camera/form data
- It does not learn from long-term real user history yet
- It is not a replacement for medical or professional nutrition advice
- The workout score target is generated from heuristic labels, not human expert labels

## 11. Report Conclusion

This model is suitable as an MVP AI planning engine for a home gym planning application. It combines Random Forest regression with rule-based safety and planning logic. The machine learning layer ranks workouts and predicts nutrition targets, while the coach layer creates a weekly schedule, meal plan, four-week progression, substitutions, safety notes, and adaptive adjustments based on user feedback.

Recommended wording for reporting:

> The system uses a hybrid AI recommendation approach. The workout model is trained using RandomForestRegressor, and the nutrition model uses MultiOutputRegressor with RandomForestRegressor. The models are trained using supervised regression on exercise and food datasets. The ML outputs are combined with rule-based safety filters and planning logic to generate personalized home gym workout schedules, meal plans, progression plans, and adaptive recovery recommendations.